## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [30]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import mlflow
from mlflow.models import infer_signature
from exercise_support import ClassSuportTransformer, MLFlowInputVariables, MLFlowRunExecution, Model
import requests
import json

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [2]:
# Implement transformation logic as in session 2

df_train = pd.read_csv("Churn_Modelling_train_test.csv")

transformer = ClassSuportTransformer()
df_transformed = transformer.transform_class_support(df_train)
df_balanced = transformer.balance_dataset(df_transformed)

df_balanced.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Geography_nan
0,674,0,45.0,7,142072.02,1,1.0,0.0,37013.29,0,0.0,0.0,0.0
1,438,1,54.0,2,0.00,1,0.0,0.0,191763.07,1,0.0,1.0,0.0
2,749,0,47.0,9,110022.74,1,0.0,1.0,135655.29,1,1.0,0.0,0.0
3,724,0,34.0,6,118235.70,2,0.0,0.0,157137.23,0,1.0,0.0,0.0
4,586,1,46.0,0,0.00,3,0.0,1.0,131553.82,1,0.0,0.0,0.0


In [6]:
import joblib

PATH = "./model_utils/"
encoder = joblib.load("./encoder.pkl")
joblib.dump(encoder, f'{PATH}one_hot_encoder.pkl')

['./model_utils/one_hot_encoder.pkl']

In [37]:
# Perform another experiment if you don't have the ones from session 2. Otherwise, this part can be skipped

# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [15]:
# Train the model and log to MLflow
model_trainer = Model()
mlflow_input = model_trainer.train_model(df_balanced)

mlflow_runner = MLFlowRunExecution()
mlflow_runner.run_mlflow_execution(mlflow_input)

2026/05/24 23:57:57 INFO mlflow.tracking.fluent: Experiment with name 'Decision Tree Experiment' does not exist. Creating a new experiment.
c:\Users\mazac\Documents\EADA\Term 3\MLOps\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'class-churn-model'.
2026/05/24 23:58:01 INFO mlflo

🏃 View run suave-shoat-13 at: http://127.0.0.1:8080/#/experiments/3/runs/66c4598ebbf6403b94358aa55ae2caf6
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/3


Created version '1' of model 'class-churn-model'.


### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [8]:
# import validation dataset to test inference
df_validation = pd.read_csv("Churn_Modelling_val.csv")
df_validation.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,8607,15694581,Rawlings,807,Spain,Male,42.0,5,0.00,2,1.0,1.0,74900.90,0
1,4685,15736963,Herring,623,France,Male,43.0,1,0.00,2,1.0,1.0,146379.30,0
2,1732,15721730,Amechi,601,Spain,Female,44.0,4,0.00,2,1.0,0.0,58561.31,0
3,4743,15762134,Liang,506,Germany,Male,59.0,8,119152.10,2,1.0,1.0,170679.74,0
4,4522,15648898,Chuang,560,Spain,Female,27.0,7,124995.98,1,1.0,1.0,114669.79,0


In [9]:
df_validation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        1001 non-null   int64  
 1   CustomerId       1001 non-null   int64  
 2   Surname          1001 non-null   object 
 3   CreditScore      1001 non-null   int64  
 4   Geography        1001 non-null   object 
 5   Gender           1001 non-null   object 
 6   Age              1001 non-null   float64
 7   Tenure           1001 non-null   int64  
 8   Balance          1001 non-null   float64
 9   NumOfProducts    1001 non-null   int64  
 10  HasCrCard        1001 non-null   float64
 11  IsActiveMember   1000 non-null   float64
 12  EstimatedSalary  1001 non-null   float64
 13  Exited           1001 non-null   int64  
dtypes: float64(5), int64(6), object(3)
memory usage: 109.6+ KB


Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [10]:
# transform data - if necessary
# Indeed needs to be transformed

transformer = ClassSuportTransformer()
df_validation = transformer.transform_class_support(df_validation)

y_validation = df_validation["Exited"]
X_validation = df_validation.drop(columns=["Exited"])

X_validation.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Geography_nan
0,807,0,42.0,5,0.00,2,1.0,1.0,74900.90,0.0,1.0,0.0
1,623,0,43.0,1,0.00,2,1.0,1.0,146379.30,0.0,0.0,0.0
2,601,1,44.0,4,0.00,2,1.0,0.0,58561.31,0.0,1.0,0.0
3,506,0,59.0,8,119152.10,2,1.0,1.0,170679.74,1.0,0.0,0.0
4,560,1,27.0,7,124995.98,1,1.0,1.0,114669.79,0.0,1.0,0.0


##### Batch Inference

In [11]:
# define a function to implement batch inference with mlflow
def batch_inference(model_uri: str, input: pd.DataFrame):
    model = mlflow.pyfunc.load_model(model_uri)
    return model.predict(input)

In [36]:
# define the model uri that should be used
model_uri = "runs:/66c4598ebbf6403b94358aa55ae2caf6/churn_model"

batch_prediction_result = batch_inference(model_uri, df_validation)

2026/05/25 00:07:07 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['Exited']`. These inputs will be ignored.


In [17]:
# check the confusion matrix
from sklearn.metrics import confusion_matrix

print("Confusion matrix:")
print(confusion_matrix(y_validation, batch_prediction_result))

Confusion matrix:
[[579 224]
 [ 62 136]]


##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [ ]:
import requests
import json

In [19]:
# import validation dataset to test inference - just one record
df_validation = pd.read_csv("Churn_Modelling_val.csv").head(1)

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [20]:
# transform data - if necessary
transformer = ClassSuportTransformer()
df_validation = transformer.transform_class_support(df_validation)

y_validation = df_validation["Exited"]
X_validation = df_validation.drop(columns=["Exited"])

X_validation.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Geography_nan
0,807,0,42.0,5,0.0,2,1.0,1.0,74900.9,0.0,1.0,0.0


In [24]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

URL = get_inference_endpoint()

In [25]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(url: str, input: pd.DataFrame):
    """Send a DataFrame to an MLflow serving endpoint using the split orientation."""
    payload = {"dataframe_split": input.to_dict(orient="split")}
    headers = {"Content-Type": "application/json"}
    return requests.post(url, data=json.dumps(payload), headers=headers)

In [31]:
response_pandas = online_inference_pandas(URL, X_validation)
print(f"Status: {response_pandas.status_code}")
print(f"Response: {response_pandas.content}")

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /invocations (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [32]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input: dict):
    headers = {"Content-Type": "application/json"}
    return requests.post(url, data=json.dumps(input), headers=headers)

In [34]:
# define the json as required by MLFlow
input_json = {
    "dataframe_split": {
        "columns": X_validation.columns.tolist(),
        "data": X_validation.values.tolist(),
    }
}
input_json

{'dataframe_split': {'columns': ['CreditScore',
   'Gender',
   'Age',
   'Tenure',
   'Balance',
   'NumOfProducts',
   'HasCrCard',
   'IsActiveMember',
   'EstimatedSalary',
   'Geography_Germany',
   'Geography_Spain',
   'Geography_nan'],
  'data': [[807.0,
    0.0,
    42.0,
    5.0,
    0.0,
    2.0,
    1.0,
    1.0,
    74900.9,
    0.0,
    1.0,
    0.0]]}}

In [35]:
response_json = online_inference_json(URL, input_json)
print(f"Status: {response_json.status_code}")
print(f"Response: {response_json.content}")

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /invocations (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))